# Habitat K Selection Plot

This notebook plots cohort-level K-means habitat clustering metrics using mean silhouette coefficient and mean Calinski-Harabasz index from K=2 to K=9.

In [1]:

# ============================================================
# 1. Imports, paths, and plotting settings
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

score_path = Path('/host/d/projects/Habitats/radiomics/habitats/cohort_mean_K2_K9_silhouette_CH_scores.xlsx')
results_out_dir = Path('/host/d/projects/Habitats/results')
results_out_dir.mkdir(parents=True, exist_ok=True)

save_pdf_path = results_out_dir / 'habitat_K_selection_SC_CH.pdf'

# ------------------------------------------------------------
# User-adjustable figure settings
# ------------------------------------------------------------

selected_k = 4

# Axis ranges chosen from the observed data range:
# SC approximately 0.290-0.410; CH approximately 6366-8463.
sc_ylim = (0.27, 0.43)
ch_ylim = (6000, 8800)

figsize = (6.4, 4.3)
sc_color = '#1f77b4'
ch_color = '#d62728'
selected_k_color = '#666666'

# ------------------------------------------------------------
# Font settings: Times New Roman
# ------------------------------------------------------------

times_font_paths = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/host/c/Windows/Fonts/timesi.ttf',
    '/host/c/Windows/Fonts/timesbi.ttf',
]
for font_path in times_font_paths:
    if os.path.isfile(font_path):
        font_manager.fontManager.addfont(font_path)

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False

print('Input score table:', score_path)
print('Output PDF:', save_pdf_path)


Input score table: /host/d/projects/Habitats/radiomics/habitats/cohort_mean_K2_K9_silhouette_CH_scores.xlsx
Output PDF: /host/d/projects/Habitats/results/habitat_K_selection_SC_CH.pdf


In [2]:

# ============================================================
# 2. Load cohort-level K selection scores
# ============================================================

if not score_path.is_file():
    raise FileNotFoundError(f'Cannot find score table: {score_path}')

score_df = pd.read_excel(score_path)

required_columns = [
    'K',
    'mean_silhouette_score',
    'mean_calinski_harabasz_score',
]
missing_columns = [col for col in required_columns if col not in score_df.columns]
if missing_columns:
    raise KeyError(f'Missing expected columns: {missing_columns}')

score_df = score_df.sort_values('K').reset_index(drop=True)

print('Score table shape:', score_df.shape)
display(score_df)

print('\nObserved ranges:')
print('SC:', float(score_df['mean_silhouette_score'].min()), 'to', float(score_df['mean_silhouette_score'].max()))
print('CH:', float(score_df['mean_calinski_harabasz_score'].min()), 'to', float(score_df['mean_calinski_harabasz_score'].max()))


Score table shape: (8, 8)


,K,mean_silhouette_score,std_silhouette_score,mean_calinski_harabasz_score,std_calinski_harabasz_score,n_cases,mean_n_voxels_for_kmeans,mean_n_features_used
0,2,0.409757,0.043572,8462.625219,8246.251552,110,11399.772727,25
1,3,0.367374,0.041607,8041.882551,7853.010315,110,11399.772727,25
2,4,0.330629,0.027677,7355.797415,7046.379995,110,11399.772727,25
3,5,0.319551,0.022668,7085.322220,6823.883073,110,11399.772727,25
4,6,0.306241,0.019052,6862.176435,6615.262831,110,11399.772727,25
5,7,0.299917,0.019312,6663.070583,6489.659847,110,11399.772727,25
6,8,0.292991,0.015492,6494.560815,6369.961292,110,11399.772727,25
7,9,0.289734,0.015270,6366.103429,6301.954427,110,11399.772727,25



Observed ranges:
SC: 0.2897339165210724 to 0.4097574055194855
CH: 6366.103429494304 to 8462.625218841797


In [3]:

# ============================================================
# 3. Plot SC and CH with two y-axes
# ============================================================

K = score_df['K'].astype(int).to_numpy()
sc = score_df['mean_silhouette_score'].astype(float).to_numpy()
ch = score_df['mean_calinski_harabasz_score'].astype(float).to_numpy()

fig, ax_sc = plt.subplots(figsize=figsize)
ax_ch = ax_sc.twinx()

# Left axis: silhouette coefficient.
line_sc, = ax_sc.plot(
    K,
    sc,
    color=sc_color,
    marker='o',
    markersize=6.5,
    linewidth=2.2,
    label='Silhouette coefficient',
)

# Right axis: Calinski-Harabasz index.
line_ch, = ax_ch.plot(
    K,
    ch,
    color=ch_color,
    marker='s',
    markersize=6.0,
    linewidth=2.2,
    label='Calinski-Harabasz index',
)

# Selected K marker.
ax_sc.axvline(
    selected_k,
    color=selected_k_color,
    linestyle='--',
    linewidth=1.2,
    alpha=0.9,
)
ax_sc.text(
    selected_k + 0.12,
    sc_ylim[1] - 0.012,
    f'Selected K = {selected_k}',
    ha='left',
    va='top',
    fontsize=11,
    color='black',
)

# Axes.
ax_sc.set_xlabel('Number of clusters K', fontsize=13, color='black')
ax_sc.set_ylabel('Mean silhouette coefficient', fontsize=13, color='black')
ax_ch.set_ylabel('Mean Calinski-Harabasz index', fontsize=13, color='black')

ax_sc.set_xlim(K.min() - 0.25, K.max() + 0.25)
ax_sc.set_xticks(K)
ax_sc.set_ylim(sc_ylim)
ax_ch.set_ylim(ch_ylim)

ax_sc.tick_params(axis='x', labelsize=12, colors='black', direction='out', length=4)
ax_sc.tick_params(axis='y', labelsize=12, colors='black', direction='out', length=4)
ax_ch.tick_params(axis='y', labelsize=12, colors='black', direction='out', length=4)

# Light grid tied to the SC axis.
ax_sc.grid(axis='y', color='0.90', linestyle='-', linewidth=0.8)
ax_sc.grid(axis='x', visible=False)
ax_sc.set_axisbelow(True)

# Spines.
for ax in [ax_sc, ax_ch]:
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_color('black')
    ax.spines['right'].set_color('black')

# Combined legend.
legend = ax_sc.legend(
    handles=[line_sc, line_ch],
    labels=['Silhouette coefficient', 'Calinski-Harabasz index'],
    loc='upper right',
    frameon=False,
    fontsize=11,
)
for text in legend.get_texts():
    text.set_color('black')

fig.tight_layout()
fig.savefig(save_pdf_path, bbox_inches='tight')
plt.close(fig)

print('Saved:', save_pdf_path)


Saved: /host/d/projects/Habitats/results/habitat_K_selection_SC_CH.pdf


## Section 2. Representative habitat visualization

This section shows a representative fixed-K habitat result for set_1/1 on slice 13. The left panel displays the original MRI with tumor mask overlay; the right panel displays a tumor-centered crop with the four habitat labels overlaid.

In [4]:

# ============================================================
# Section 2. Representative habitat visualization
# ============================================================

import os
from pathlib import Path

import numpy as np
import nibabel as nb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import font_manager

# ------------------------------------------------------------
# User-adjustable settings
# ------------------------------------------------------------

patient_set = 'set_1'
patient_index = 1
slice_index = 13  # 0-based index; slice 13 means the 14th slice.

window_level = 250
window_width = 500

tumor_mask_alpha = 0.45
habitat_alpha = 1.0

# Rotate display if needed. k=1 has been visually useful for axial MRI in this project.
display_rot90_k = 1
display_flipud = True

# Crop design: approximate tumor span should occupy around 70% of the crop extent.
tumor_target_fraction = 0.70
crop_extra_scale = 1.05

# Output paths
results_out_dir = Path('/host/d/projects/Habitats/results')
results_out_dir.mkdir(parents=True, exist_ok=True)
representative_pdf_path = results_out_dir / 'habitat_representative_set_1_1_slice13.pdf'
original_overlay_pdf_path = results_out_dir / 'habitat_representative_original_overlay_set_1_1_slice13.pdf'
habitat_crop_pdf_path = results_out_dir / 'habitat_representative_habitat_crop_set_1_1_slice13.pdf'

# Input paths
original_data_root = Path('/host/e/D/Data/Habitats/Jishuitan/original_data')
habitat_root = Path('/host/d/projects/Habitats/radiomics/habitats')

img_path = original_data_root / patient_set / str(patient_index) / 'img.nii.gz'
label_path = original_data_root / patient_set / str(patient_index) / 'label.nii.gz'
habitat_path = habitat_root / patient_set / str(patient_index) / 'habitats_original_space_final.nii.gz'

# ------------------------------------------------------------
# Font settings
# ------------------------------------------------------------

times_font_paths = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/host/c/Windows/Fonts/timesi.ttf',
    '/host/c/Windows/Fonts/timesbi.ttf',
]
for font_path in times_font_paths:
    if os.path.isfile(font_path):
        font_manager.fontManager.addfont(font_path)

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False

print('Image:', img_path)
print('Label:', label_path)
print('Habitat:', habitat_path)
print('Output combined:', representative_pdf_path)


Image: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/img.nii.gz
Label: /host/e/D/Data/Habitats/Jishuitan/original_data/set_1/1/label.nii.gz
Habitat: /host/d/projects/Habitats/radiomics/habitats/set_1/1/habitats_original_space_final.nii.gz
Output combined: /host/d/projects/Habitats/results/habitat_representative_set_1_1_slice13.pdf


In [5]:

# ============================================================
# Load image, tumor mask, and habitat mask
# ============================================================

for p in [img_path, label_path, habitat_path]:
    if not p.is_file():
        raise FileNotFoundError(f'Missing file: {p}')

img_arr = nb.load(str(img_path)).get_fdata()
label_arr = nb.load(str(label_path)).get_fdata()
habitat_arr = nb.load(str(habitat_path)).get_fdata()

if not (img_arr.shape == label_arr.shape == habitat_arr.shape):
    raise RuntimeError(
        f'Shape mismatch: img={img_arr.shape}, label={label_arr.shape}, habitat={habitat_arr.shape}'
    )

if slice_index < 0 or slice_index >= img_arr.shape[2]:
    raise IndexError(f'slice_index={slice_index} is outside image z range 0-{img_arr.shape[2]-1}')

img_slice = img_arr[:, :, slice_index]
label_slice = label_arr[:, :, slice_index] > 0
habitat_slice = np.rint(habitat_arr[:, :, slice_index]).astype(int)

if display_rot90_k is not None:
    img_slice = np.rot90(img_slice, k=display_rot90_k)
    label_slice = np.rot90(label_slice, k=display_rot90_k)
    habitat_slice = np.rot90(habitat_slice, k=display_rot90_k)

if display_flipud:
    img_slice = np.flipud(img_slice)
    label_slice = np.flipud(label_slice)
    habitat_slice = np.flipud(habitat_slice)

# Keep habitat labels only inside the original tumor mask for display.
habitat_slice = np.where(label_slice, habitat_slice, 0)

print('Image shape:', img_arr.shape)
print('Slice index:', slice_index)
print('Tumor pixels on slice:', int(label_slice.sum()))
print('Habitat labels on slice:', np.unique(habitat_slice[habitat_slice > 0]).tolist())
print('WL/WW:', window_level, window_width)


Image shape: (1024, 1024, 23)
Slice index: 13
Tumor pixels on slice: 70815
Habitat labels on slice: [1, 2, 3, 4]
WL/WW: 250 500


In [6]:

# ============================================================
# Plot helper functions
# ============================================================

vmin = window_level - window_width / 2.0
vmax = window_level + window_width / 2.0

# Tumor mask: red, semi-transparent.
tumor_cmap = mcolors.ListedColormap(['#ff0033'])

# Habitat mask: categorical, non-red colors for labels 1-4.
habitat_colors = [
    '#1f77b4',  # habitat 1: blue
    '#2ca02c',  # habitat 2: green
    '#ffbf00',  # habitat 3: amber
    '#9467bd',  # habitat 4: purple
]
habitat_cmap = mcolors.ListedColormap(habitat_colors)
habitat_norm = mcolors.BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5], habitat_cmap.N)


def crop_or_pad_2d(arr, center_x, center_y, crop_h, crop_w, pad_value=0):
    arr = np.asarray(arr)
    h, w = arr.shape

    x0 = int(round(center_x - crop_h / 2))
    y0 = int(round(center_y - crop_w / 2))
    x1 = x0 + crop_h
    y1 = y0 + crop_w

    src_x0 = max(0, x0)
    src_y0 = max(0, y0)
    src_x1 = min(h, x1)
    src_y1 = min(w, y1)

    dst_x0 = src_x0 - x0
    dst_y0 = src_y0 - y0
    dst_x1 = dst_x0 + (src_x1 - src_x0)
    dst_y1 = dst_y0 + (src_y1 - src_y0)

    out = np.full((crop_h, crop_w), pad_value, dtype=arr.dtype)
    out[dst_x0:dst_x1, dst_y0:dst_y1] = arr[src_x0:src_x1, src_y0:src_y1]
    return out, (x0, x1, y0, y1)


def compute_tumor_center_crop(label_2d):
    coords = np.where(label_2d > 0)
    if len(coords[0]) == 0:
        raise RuntimeError('No tumor pixels on the selected slice.')

    x_min, x_max = int(coords[0].min()), int(coords[0].max())
    y_min, y_max = int(coords[1].min()), int(coords[1].max())

    bbox_h = x_max - x_min + 1
    bbox_w = y_max - y_min + 1
    center_x = (x_min + x_max) / 2.0
    center_y = (y_min + y_max) / 2.0

    crop_size = int(np.ceil(max(bbox_h, bbox_w) / tumor_target_fraction * crop_extra_scale))
    crop_size = max(crop_size, bbox_h, bbox_w)

    return center_x, center_y, crop_size, crop_size, {
        'x_min': x_min,
        'x_max': x_max,
        'y_min': y_min,
        'y_max': y_max,
        'bbox_h': bbox_h,
        'bbox_w': bbox_w,
        'crop_h': crop_size,
        'crop_w': crop_size,
    }


def plot_original_overlay(ax):
    ax.imshow(img_slice, cmap='gray', vmin=vmin, vmax=vmax, interpolation='nearest')
    masked_label = np.ma.masked_where(~label_slice, label_slice)
    ax.imshow(masked_label, cmap=tumor_cmap, alpha=tumor_mask_alpha, interpolation='nearest')
    ax.set_axis_off()


def plot_habitat_crop(ax):
    center_x, center_y, crop_h, crop_w, crop_info = compute_tumor_center_crop(label_slice)
    img_crop, crop_box = crop_or_pad_2d(img_slice, center_x, center_y, crop_h, crop_w, pad_value=0)
    habitat_crop, _ = crop_or_pad_2d(habitat_slice, center_x, center_y, crop_h, crop_w, pad_value=0)

    ax.imshow(img_crop, cmap='gray', vmin=vmin, vmax=vmax, interpolation='nearest')
    habitat_masked = np.ma.masked_where(habitat_crop <= 0, habitat_crop)
    ax.imshow(habitat_masked, cmap=habitat_cmap, norm=habitat_norm, alpha=habitat_alpha, interpolation='nearest')
    ax.set_axis_off()

    crop_info['crop_box_x0_x1_y0_y1'] = crop_box
    crop_info['habitat_labels_in_crop'] = np.unique(habitat_crop[habitat_crop > 0]).tolist()
    return crop_info

print('Plot helpers ready.')


Plot helpers ready.


In [7]:

# ============================================================
# Generate representative habitat visualization
# ============================================================

# Separate original MRI + tumor mask panel.
fig, ax = plt.subplots(figsize=(3.2, 3.2))
plot_original_overlay(ax)
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
fig.savefig(original_overlay_pdf_path, bbox_inches='tight', pad_inches=0)
plt.close(fig)
print('Saved original overlay:', original_overlay_pdf_path)

# Separate cropped MRI + habitat mask panel.
fig, ax = plt.subplots(figsize=(3.2, 3.2))
crop_info = plot_habitat_crop(ax)
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
fig.savefig(habitat_crop_pdf_path, bbox_inches='tight', pad_inches=0)
plt.close(fig)
print('Saved habitat crop:', habitat_crop_pdf_path)

# Combined 1x2 panel.
fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.2), gridspec_kw={'wspace': 0.03})
plot_original_overlay(axes[0])
crop_info = plot_habitat_crop(axes[1])
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, wspace=0.03)
fig.savefig(representative_pdf_path, bbox_inches='tight', pad_inches=0.02)
plt.close(fig)

print('Saved combined representative figure:', representative_pdf_path)
print('Crop info:', crop_info)


Saved original overlay: /host/d/projects/Habitats/results/habitat_representative_original_overlay_set_1_1_slice13.pdf
Saved habitat crop: /host/d/projects/Habitats/results/habitat_representative_habitat_crop_set_1_1_slice13.pdf
Saved combined representative figure: /host/d/projects/Habitats/results/habitat_representative_set_1_1_slice13.pdf
Crop info: {'x_min': 406, 'x_max': 727, 'y_min': 613, 'y_max': 890, 'bbox_h': 322, 'bbox_w': 278, 'crop_h': 484, 'crop_w': 484, 'crop_box_x0_x1_y0_y1': (324, 808, 510, 994), 'habitat_labels_in_crop': [1, 2, 3, 4]}
